In [1]:
%run ./../notebook_init.py

import os
import uproot

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from core import DATA_FOLDER, RESULTS_FOLDER
from scripts.connie_training_utils import (Seed, extract_skeleton_features,
                                           extract_fourier_descriptors)

In [2]:
train_data = os.path.join(DATA_FOLDER, "train_data_root_full")
test_data = os.path.join(DATA_FOLDER, "test_data_root")

seed = Seed()

categories = ["Blob", "Diffusion Hit", "Electron", "Muon", "Others"]

branch_name = "hitSumm"

In [3]:
train_all_data_list = []
test_all_data_list = []


print("Starting data loading")
for category in categories:
    train_category_path = os.path.join(train_data, category)
    train_root_files = glob(os.path.join(train_category_path, "*.root"))

    if not train_root_files:
        print(f"Warning: No .root files found in {train_category_path}")
        continue

    test_category_path = os.path.join(test_data, category)
    test_root_files = glob(os.path.join(test_category_path, "*.root"))

    if not test_root_files:
        print(f"Warning: No .root files found in {test_category_path}")
        continue

    print(f"TRAIN - Processing category: {category} ({len(train_root_files)} files)")
    for idx, file_path in enumerate(train_root_files):
        try:
            with uproot.open(file_path) as file:
                if branch_name not in file:
                    print(f"Warning: TTree '{branch_name}' not found in {file_path}. Skipping.")
                    continue
                train_file_branch = file[branch_name]
                df_train = train_file_branch.arrays(library="pd")
                df_train['label'] = category
                train_all_data_list.append(df_train)

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

    print(f"TEST - Processing category: {category} ({len(test_root_files)} files)")
    for idx, file_path in enumerate(test_root_files):
        try:
            with uproot.open(file_path) as file:
                if branch_name not in file:
                    print(f"Warning: TTree '{branch_name}' not found in {file_path}. Skipping.")
                    continue
                test_file_branch = file[branch_name]
                df_test = test_file_branch.arrays(library="pd")
                df_test['label'] = category
                test_all_data_list.append(df_test)

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

Starting data loading
TRAIN - Processing category: Blob (351 files)
TEST - Processing category: Blob (62 files)
TRAIN - Processing category: Diffusion Hit (44 files)
TEST - Processing category: Diffusion Hit (8 files)
TRAIN - Processing category: Electron (366 files)
TEST - Processing category: Electron (65 files)
TRAIN - Processing category: Muon (2596 files)
TEST - Processing category: Muon (458 files)
TRAIN - Processing category: Others (220 files)
TEST - Processing category: Others (39 files)


Combine all DataFrames into a single DataFrame

In [4]:
if train_all_data_list:
    train_df_combined = pd.concat(train_all_data_list, ignore_index=True)
    print(f"Successfully loaded {len(train_df_combined)} rows of data.")
else:
    print("No data loaded.")

if test_all_data_list:
    test_df_combined = pd.concat(test_all_data_list, ignore_index=True)
    print(f"Successfully loaded {len(test_df_combined)} rows of data.")
else:
    print("No data loaded.")

Successfully loaded 3577 rows of data.
Successfully loaded 632 rows of data.


In [5]:
train_df_processed = train_df_combined.copy()

train_df_processed["ePixMean"] = train_df_processed["ePix"].apply(np.mean)
train_df_processed["levelMean"] = train_df_processed["level"].apply(np.mean)


test_df_processed = test_df_combined.copy()

test_df_processed["ePixMean"] = test_df_processed["ePix"].apply(np.mean)
test_df_processed["levelMean"] = test_df_processed["level"].apply(np.mean)



In [6]:
skeleton_features_train = train_df_processed.apply(
    lambda row: extract_skeleton_features(row['xPix'],
                                          row['yPix']),
    axis=1)

fd_features_train = train_df_processed.apply(
    lambda row: extract_fourier_descriptors(row['xPix'],
                                            row['yPix'], 10),
    axis=1)

skeleton_features_df_train = pd.json_normalize(skeleton_features_train)
fd_features_df_train = pd.json_normalize(fd_features_train)

train_df_processed = pd.concat([
    train_df_processed.reset_index(drop=True),
    skeleton_features_df_train,
    fd_features_df_train,
], axis=1)

# Test data
skeleton_features_test = test_df_processed.apply(
    lambda row: extract_skeleton_features(row['xPix'],
                                          row['yPix']),
    axis=1)

fd_features_test = test_df_processed.apply(
    lambda row: extract_fourier_descriptors(row['xPix'],
                                            row['yPix'], 10),
    axis=1)


skeleton_features_df_test = pd.json_normalize(skeleton_features_test)
fd_features_df_test = pd.json_normalize(fd_features_test)

test_df_processed = pd.concat([
    test_df_processed.reset_index(drop=True),
    skeleton_features_df_test,
    fd_features_df_test,
], axis=1)


In [7]:
train_df_processed = train_df_processed.drop(columns=["label", "xPix", "yPix",
                                                      "level", "ePix", "flag"])

# Drop columns with no variance
train_df_processed = train_df_processed.loc[:, train_df_processed.nunique() > 1]


test_df_processed = test_df_processed.drop(columns=["label", "xPix", "yPix",
                                                  "level", "ePix", "flag"])

# Drop columns with no variance
test_df_processed = test_df_processed.loc[:, test_df_processed.nunique() > 1]


Removing features from the dataframe

In [8]:
drop_cols = [
    "ohdu", "chid", "skpID", "runID", "imgID", "Gain", "expoStart",
    "DeltaT", "NpixAC", "E1", "n1", "xBary1", "yBary1",
    "xVar1", "yVar1", "nSavedPix", "nxPix", "nyPix",
    "nlevel", "nePix", "xMin", "xMax", "yMin", "yMax",
    "skeleton_length", "branch_to_end_ratio"]


train_df_processed_final = train_df_processed.drop(columns=drop_cols)

test_df_processed_final = test_df_processed.drop(columns=drop_cols)

In [9]:
print(test_df_processed_final.columns)
print(train_df_processed_final.columns)

Index(['imgG', 'Noise', 'SER', 'ExpTime', 'E0', 'n0', 'xBary0', 'yBary0',
       'xVar0', 'yVar0', 'EventID', 'ePixMean', 'levelMean', 'num_branches',
       'num_endpoints', 'skeleton_area_ratio', 'skeleton_tortuosity',
       'skeleton_linearity_deviation', 'skeleton_curvature_sum', 'FD_2',
       'FD_3', 'FD_4', 'FD_5', 'FD_6', 'FD_7', 'FD_8', 'FD_9', 'FD_10',
       'FD_11'],
      dtype='object')
Index(['imgG', 'Noise', 'SER', 'ExpTime', 'E0', 'n0', 'xBary0', 'yBary0',
       'xVar0', 'yVar0', 'EventID', 'ePixMean', 'levelMean', 'num_branches',
       'num_endpoints', 'skeleton_area_ratio', 'skeleton_tortuosity',
       'skeleton_linearity_deviation', 'skeleton_curvature_sum', 'FD_2',
       'FD_3', 'FD_4', 'FD_5', 'FD_6', 'FD_7', 'FD_8', 'FD_9', 'FD_10',
       'FD_11'],
      dtype='object')


In [10]:
label_encoder = LabelEncoder()

X_train_split = train_df_processed_final.copy()
X_test_split = test_df_processed_final.copy()


scaler = StandardScaler()
X_train_split = scaler.fit_transform(X_train_split)
X_test_split = scaler.transform(X_test_split)

y_train_split = label_encoder.fit_transform(train_df_combined["label"])
y_test_split = label_encoder.transform(test_df_combined["label"])

for i, class_name in enumerate(label_encoder.classes_):
    print(f"Class ID {i}: {class_name}")


Class ID 0: Blob
Class ID 1: Diffusion Hit
Class ID 2: Electron
Class ID 3: Muon
Class ID 4: Others


In [11]:
#xgboost

# Trial 85
best_params_xgb_multiclass = {
    "eval_metric": "mlogloss",
    "gamma": 0.719283932509348,  
    "learning_rate": 0.1216873848492071,
    "max_depth": 12,
    "num_class": 5,
    "n_estimators": 181,
    "objective": "multi:softprob",
    "random_state": 42,
    "use_label_encoder": False
}

In [13]:
#random_forest
# Trial 93
best_params_rf_multiclass = {
    "class_weight": "balanced",
    "max_features": 0.24534156911868338,
    "min_samples_leaf": 3,
    "min_samples_split": 10,
    "n_estimators": 408,
    "random_state": 42
}


In [14]:
# Save metrics
metrics_dir_rf = os.path.join(RESULTS_FOLDER, "test_metrics_rf_multiclass_img_descriptor_new_features")
metrics_dir_xgb = os.path.join(RESULTS_FOLDER, "test_metrics_xgboost_multiclass_img_descriptor_new_features")

os.makedirs(metrics_dir_rf, exist_ok=True)
os.makedirs(metrics_dir_xgb, exist_ok=True)

Evaluation on Test dataset

In [15]:
def test_model(is_xgboost=True):
    if is_xgboost:
        model_class = XGBClassifier
        best_params_multiclass = best_params_xgb_multiclass
        metrics_dir = metrics_dir_xgb
    else:
        model_class = RandomForestClassifier
        best_params_multiclass = best_params_rf_multiclass
        metrics_dir = metrics_dir_rf
    
    model = model_class(**best_params_multiclass)
    
    if is_xgboost:
        classes = np.unique(y_train_split)
        class_weights = compute_class_weight('balanced', classes=classes, y=y_train_split)
        weight_dict = dict(zip(classes, class_weights))
        sample_weights = np.array([weight_dict[y] for y in y_train_split])
        model.fit(X_train_split, y_train_split, sample_weight=sample_weights)
    else:
        model.fit(X_train_split, y_train_split)
    
    preds_int = model.predict(X_test_split)
    probas = model.predict_proba(X_test_split)
    
    preds_str = label_encoder.inverse_transform(preds_int)
    
    report = classification_report(
        y_test_split,
        preds_int,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report).transpose()
    
    cm = confusion_matrix(y_test_split,
                          preds_int,
                          labels=np.arange(len(label_encoder.classes_)))
    
    report_path = os.path.join(metrics_dir, "multiclass_classification_report.csv") 
    report_df.to_csv(report_path)
    
    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()
    cm_path = os.path.join(metrics_dir, "multiclass_confusion_matrix.png")
    fig.savefig(cm_path)
    plt.close(fig)

    cm_norm = confusion_matrix(
        y_test_split,
        preds_int,
        labels=np.arange(len(label_encoder.classes_)),
        normalize='true'
    )
    
    fig_norm, ax_norm = plt.subplots(figsize=(8,6))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".3f",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
        ax=ax_norm
    )
    
    ax_norm.set_xlabel("Predicted")
    ax_norm.set_ylabel("True")
    #ax_norm.set_title("Normalized Confusion Matrix")
    
    fig_norm.tight_layout()
    
    cm_norm_path = os.path.join(
        metrics_dir,
        "multiclass_confusion_matrix_normalized.png"
    )
    
    fig_norm.savefig(cm_norm_path, dpi=300, bbox_inches='tight')
    plt.close(fig_norm)

    return model, report_df, cm, probas

In [16]:
xgb_model, xgb_report, xgb_cm, xgb_probas = test_model(is_xgboost=True)
rf_model, rf_report, rf_cm, rf_probas = test_model(is_xgboost=False)

In [17]:
print("\n" + "="*60)
print("XGBoost Multiclass Test Results")
print("="*60)
print(f"Accuracy:    {xgb_report.loc['accuracy', 'precision']:.4f}")
print(f"Precision:   {xgb_report.loc['macro avg', 'precision']:.4f}")
print(f"Recall:      {xgb_report.loc['macro avg', 'recall']:.4f}")
print(f"Macro F1:    {xgb_report.loc['macro avg', 'f1-score']:.4f}")

print("\n" + "="*60)
print("RandomForest Multiclass Test Results")
print("="*60)
print(f"Accuracy:    {rf_report.loc['accuracy', 'precision']:.4f}")
print(f"Precision:   {rf_report.loc['macro avg', 'precision']:.4f}")
print(f"Recall:      {rf_report.loc['macro avg', 'recall']:.4f}")
print(f"Macro F1:    {rf_report.loc['macro avg', 'f1-score']:.4f}")



XGBoost Multiclass Test Results
Accuracy:    0.9114
Precision:   0.8151
Recall:      0.8695
Macro F1:    0.8404

RandomForest Multiclass Test Results
Accuracy:    0.9051
Precision:   0.7901
Recall:      0.8446
Macro F1:    0.8119
